# 01 Data Pipeline

Runs the extraction pipeline end to end and prepares the artifacts that training reads:

1. `src/extract_midi.py` copies MIDI files from the raw archive into `data/interim/<composer>/`.
2. `src/extract_roll.py` writes one piano roll npz per song plus `rolls_manifest.csv`.
3. `src/extract_features.py` writes the 39 column music theory feature table.
4. `src/make_splits.py` assigns every song to one of 5 stratified cross validation folds.

Each extractor is skipped when its output already exists (set `FORCE = True` to rerun; roll
extraction is the slow step). Gate cells assert the invariants the modeling code depends on,
so a clean run of this notebook means the data is ready for `src/train.py`. No scaling or
imputation happens here: those are fit per fold on training songs only, inside train.py.

In [1]:
import subprocess
from pathlib import Path

import pandas as pd

PYTHON = "/opt/miniconda3/envs/composer-classification/bin/python"
REPO_ROOT = Path.cwd().parent
FORCE = False  # True reruns every extractor even if its output exists


def run(script):
    """Run one src script from the repo root and echo its summary output."""
    result = subprocess.run([PYTHON, f"src/{script}"], cwd=REPO_ROOT,
                            capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"{script} failed:\n{result.stderr}")

## Extraction

Each step runs only if its output is missing (or `FORCE` is set). The printed summaries are
the extractors' own gate output: file counts per composer and skip counts.

In [2]:
steps = [
    ("extract_midi.py", REPO_ROOT / "data/interim"),
    ("extract_roll.py", REPO_ROOT / "data/processed/rolls_manifest.csv"),
    ("extract_features.py", REPO_ROOT / "data/processed/features.csv"),
]
for script, output in steps:
    if FORCE or not output.exists():
        print(f"running {script} ...")
        run(script)
    else:
        print(f"skipping {script}: {output.relative_to(REPO_ROOT)} exists")

skipping extract_midi.py: data/interim exists
skipping extract_roll.py: data/processed/rolls_manifest.csv exists
skipping extract_features.py: data/processed/features.csv exists


## Gate checks

The invariants the loaders and splits rely on: both artifacts cover the identical 1,628 song
set keyed by filename and composer, class counts match the source archive, and the only NaNs
are the two monophonic songs where `vi_dissonance` is undefined by policy.

In [3]:
manifest = pd.read_csv(REPO_ROOT / "data/processed/rolls_manifest.csv")
features = pd.read_csv(REPO_ROOT / "data/processed/features.csv")

assert len(manifest) == 1628, len(manifest)
assert len(features) == 1628, len(features)

key = ["filename", "composer"]
assert set(map(tuple, manifest[key].values)) == set(map(tuple, features[key].values))

counts = manifest["composer"].value_counts()
assert counts.to_dict() == {"bach": 1024, "beethoven": 212, "chopin": 136, "mozart": 256}

na = features.isna().sum()
na = na[na > 0]
assert na.to_dict() == {"vi_dissonance": 2}, na.to_dict()

print(f"{len(manifest)} songs in both artifacts, identical key sets")
print(counts.to_string())
print(f"n_frames: min {manifest['n_frames'].min()}, "
      f"median {manifest['n_frames'].median():.0f}, max {manifest['n_frames'].max()}")
print(f"NaNs: {na.to_dict()} (monophonic songs, imputed per fold in train.py)")

1628 songs in both artifacts, identical key sets
composer
bach         1024
mozart        256
beethoven     212
chopin        136
n_frames: min 175, median 1356, max 52094
NaNs: {'vi_dissonance': 2} (monophonic songs, imputed per fold in train.py)


## Cross validation folds

`make_splits.py` reads the manifest and assigns each song to one of 5 folds with
`StratifiedKFold` (seeded, shuffled), so reruns reproduce the identical file. Splits are by
song: a song's roll and feature vector always land on the same side of any split.

In [4]:
run("make_splits.py")

splits = pd.read_csv(REPO_ROOT / "data/processed/splits.csv")
assert len(splits) == 1628
crosstab = pd.crosstab(splits["fold"], splits["composer"])
assert (crosstab > 0).all().all(), "every fold needs every composer"
assert crosstab.sum(axis=1).max() - crosstab.sum(axis=1).min() <= 1
assert (crosstab.max() - crosstab.min() <= 1).all(), "per composer counts differ by > 1"
crosstab

wrote data/processed/splits.csv: 1628 rows, 5 folds, seed 42
composer  bach  beethoven  chopin  mozart
fold                                     
0          205         43      27      51
1          205         42      28      51
2          205         42      27      52
3          205         42      27      51
4          204         43      27      51



composer,bach,beethoven,chopin,mozart
fold,,,,
0,205,43,27,51
1,205,42,28,51
2,205,42,27,52
3,205,42,27,51
4,204,43,27,51


## Takeaways

- All 1,628 songs pass through both extractors with identical key sets, so the roll and
  feature branches of the model always describe the same song. Composer counts (Bach 1024,
  Beethoven 212, Chopin 136, Mozart 256) confirm the class imbalance that motivates class
  weights and macro-F1.
- Song lengths span 175 to 52,094 frames, so training needs the fixed 300 frame crop and
  evaluation needs window averaging; both are implemented in `src/dataset.py`.
- The 5 folds are balanced to within one song per composer, so every fold can serve as a
  held out set for macro-F1 without starving Chopin (27 to 28 songs per fold).
- The data side is done. Next step is `src/train.py`, which fits preprocessing per fold and
  trains the CNN/LSTM fusion model.